In [14]:
"""
Derby Maker Template (v3)

Purpose:
- Define a Prophet's DAG.
- Add regression-table models as StudyModel objects.
- Add one Moment per usable reported coefficient.
- Run God's DAG v2 on those moments.

This file is intentionally blank with respect to studies/models.
Use it as a fill-in template.
"""

from gods_dag_v2 import DAGSpec, Edge, StudyModel, Moment, GodsDagEngine
import numpy as np

# ADD THESE TWO LINES:
import importlib, gods_dag_v2
importlib.reload(gods_dag_v2)
from gods_dag_v2 import DAGSpec, Edge, StudyModel, Moment, GodsDagEngine  # re-import after reload

np.random.seed(42)


# ============================================================
# STEP 1: DEFINE THE PROPHET'S DAG
# ------------------------------------------------------------
# Replace this default DAG with your own theory if needed.
# The example below is a reduced-form conflict setup.
# ============================================================
dag = DAGSpec(
    nodes=[
        'Capabilities',
        'Trade',
        'TerritorialDispute',
        'CommitmentProblem',
        'PreferenceSimilarity',
        'LeaderHorizon',
        'Democracy',
        'Conflict',
        
    ],
    edges=[
        Edge('Capabilities',       'Conflict', sign=+1, initial_value=0.4),
        Edge('Trade',              'Conflict', sign=-1, initial_value=-0.4),
        Edge('TerritorialDispute', 'Conflict', sign=+1, initial_value=0.8),
        Edge('CommitmentProblem',  'Conflict', sign=+1, initial_value=0.6),
        Edge('PreferenceSimilarity', 'Conflict', sign=-1, initial_value=-0.3),  # flipped sign
        Edge('LeaderHorizon',      'Conflict', sign=-1, initial_value=-0.3),
        Edge('Democracy',          'Conflict', sign=-1, initial_value=-0.2),
    ],
    primitives=['Capabilities', 'Trade', 'TerritorialDispute', 'CommitmentProblem', 'PreferenceSimilarity', 'LeaderHorizon'],
    confounders=['Democracy'],  # GDP removed from confounders too
    outcome_node='Conflict',
)

# ============================================================
# STEP 2: DEFINE ALIASES / CANONICAL NAMES
# ------------------------------------------------------------
# Conservative aliasing only:
# - Collapse spelling/format variants
# - DO NOT collapse different constructs
# - DO NOT collapse transforms/interactions into plain variables
# - Map to broader DAG nodes later in node_map, not here
# ============================================================
ALIASES = {
    # ----------------------------
    # democracy
    # ----------------------------
    "jointdemo": "joint democracy",
    "democratic dyad": "joint democracy",
    "both democracies": "joint democracy",

    "democracy low": "democracy (low)",
    "lower of democracy scores": "democracy (low)",

    "initiator democracy score": "initiator democracy",
    "sender democracy": "sender democracy",
    "target democracy": "target democracy",
    "pt's democracy": "target democracy",
    "target's polity": "target polity",
    "state a is a democracy (polity iv)": "state a democracy",
    "state b is a democracy (polity iv)": "state b democracy",
    "democracyl": "democracy",

    # ----------------------------
    # alliance
    # ----------------------------
    "allied": "alliance",
    "ally": "alliance",
    "allies": "alliance",
    "joint alliance": "alliance",
    "countries are allied": "alliance",
    "defense pact": "alliance",
    "defensive alliance t-1": "alliance",
    "alliance portfolio similarity": "alliance portfolio similarity",

    # ----------------------------
    # capabilities / power balance
    # keep transforms distinct where possible
    # ----------------------------
    "capabilities ratio": "capability ratio",
    "relative capabilities": "relative capabilities",
    "target relative capabilities": "target relative capabilities",
    "capability ratio (log)": "log capability ratio",
    "ln(capability ratio)": "log capability ratio",
    "capratio": "capability ratio",
    "relpow": "relative power",
    "relative power": "relative power",
    "stronger state's capability": "stronger state capability",
    "power ratio": "power ratio",

    # ----------------------------
    # trade / interdependence
    # keep distinct families distinct
    # ----------------------------
    "log trade": "log trade",
    "trade dependency": "trade dependence",
    "dependencyl": "trade dependence",
    "economic interdependence": "economic interdependence",
    "same trading community": "same trading community",
    "trade": "trade",
    "pt's trade openness": "trade openness",
    "pi's unrest×pt's trade openness": "unrest × trade openness",

    # ----------------------------
    # territorial / issue variables
    # do not collapse rivalry into dispute
    # ----------------------------
    "territorial disputes": "territorial dispute",
    "territory": "territorial dispute",
    "territorial rivalry (strategic)": "territorial rivalry strategic",
    "territorial rivalry (kgd)": "territorial rivalry kgd",

    # ----------------------------
    # commitment / uncertainty / power shift
    # keep observables distinct here
    # ----------------------------
    "expected shift in the distribution of power": "expected power shift",
    "dyadic difference": "dyadic difference",
    "systemic diff.": "systemic difference",
    "dyadic × systemic": "dyadic × systemic",
    "power parity": "power parity",
    "dyadic uncertainty": "dyadic uncertainty",
    "system uncertainty": "system uncertainty",

    # ----------------------------
    # rivalry history
    # ----------------------------
    "strategic rivalry": "strategic rivalry",
    "kgd rivalry": "kgd rivalry",
    "cold war rival": "cold war rival",
    "past dispute": "past dispute",
    "past mids": "past mids",
    "involved in interstate rivalry": "interstate rivalry",

    # ----------------------------
    # distance / contiguity
    # keep logged forms distinct
    # ----------------------------
    "distance (ln)": "log distance",
    "distance (logged)": "log distance",
    "distance (log)": "log distance",
    "ln(distance)": "log distance",
    "ln(distance between states)": "log distance between states",
    "log distance": "log distance",
    "log(distance)": "log distance",
    "lndistance": "log distance",
    "capital-to-capital distance": "capital-to-capital distance",
    "distance between capitals (logged)": "log capital distance",
    "log of distance between capital cities, km": "log capital distance",
    "not contiguous": "noncontiguity",
    "noncontiguity": "noncontiguity",
    "border": "contiguity",
    "countries directly contiguous": "contiguity",

    # ----------------------------
    # peace horizon / settlement
    # keep peace-years family and settlement family distinct
    # ----------------------------
    "age at first mid": "age at first mid",
    "border age": "border age",
    "brevity of peace (5-year half-life)": "brevity of peace",
    "settled border": "settled border",
    "previously settled border": "previously settled border",
    "border treaty since last mid": "border treaty since last MID",
    "peaceful transfer": "peaceful transfer",
    "settlement": "settlement",

    # ----------------------------
    # preferences / alignment
    # keep SIMILARITY semantics explicit
    # ----------------------------
    "s score": "s-score",
    "dyadic s-score": "s-score",
    "un vote similarity": "un vote similarity",
    "similarity in foreign policy interests": "foreign policy similarity",
    "interests": "foreign policy interests",
    "joint igo membership": "joint igo membership",
    "foreign policy similarity": "foreign policy similarity",
}


def canon(x: str) -> str:
    if x is None:
        return x
    x = x.strip().lower()
    return ALIASES.get(x, x)


def canon_list(xs):
    return [canon(x) for x in xs]










# ============================================================
# STEP 3: DEFINE STUDY MODELS
# ------------------------------------------------------------
# New rule:
# - controls come from the exact study's conditioning_set
# - moments are hero-first, conservative
# - no pooled stand-in coefficients
# - no interaction moments
# ============================================================

import json

with open("/Users/kapurao/Downloads/hazard_library_v4.json", "r") as f:
    HLIB = json.load(f)


def get_study_entry(study_name: str) -> dict:
    if study_name not in HLIB:
        raise KeyError(f"Study not found in hazard library v4: {study_name}")
    return HLIB[study_name]


def get_var_entry(study_name: str, var_name: str) -> dict:
    entry = get_study_entry(study_name)
    vars_dict = entry["variables"]
    if var_name not in vars_dict:
        raise KeyError(f"Variable '{var_name}' not found in study '{study_name}'")
    return vars_dict[var_name]


def make_moment(study_name: str, var_name: str, role: str = "hero", note: str = "") -> Moment:
    v = get_var_entry(study_name, var_name)
    return Moment(
        variable=canon(v["variable_name"]),
        beta_hat=float(v["beta"]),
        se=float(v["se"]),
        role=role,
        admissibility="candidate",
        note=note,
    )


def make_study_model(
    study_name: str,
    hero_vars: list[str],
    node_map: dict,
    extra_moments: list[tuple[str, str, str]] | None = None,  # (var_name, role, note)
    outcome: str = "Conflict",
    estimator: str = "cox",
) -> StudyModel:
    entry = get_study_entry(study_name)

    moments = [make_moment(study_name, hv, role="hero") for hv in hero_vars]

    if extra_moments:
        for var_name, role, note in extra_moments:
            moments.append(make_moment(study_name, var_name, role=role, note=note))

    return StudyModel(
        name=study_name,
        outcome=outcome,
        estimator=estimator,
        controls=canon_list(entry["conditioning_set"]),
        node_map=node_map,
        moments=moments,
    )


study_models = []

# ------------------------------------------------------------
# 1. Gartzke Weisiger 2013
# Keep capability ratio only.
# Drop dyadic × systemic as a moment.
# ------------------------------------------------------------
study_models.append(
    make_study_model(
        study_name="Gartzke Weisiger 2013",
        hero_vars=["capability ratio"],
        node_map={
            canon("capability ratio"): "Capabilities",
            canon("democracy (low)"): "Democracy",
            canon("dyadic difference"): "CommitmentProblem",
            canon("systemic diff."): "CommitmentProblem",
            "Conflict": "Conflict",
        },
        extra_moments=[
            ("dyadic difference", "control", "same-study anchor; exact table coefficient"),
            ("systemic diff.", "control", "same-study anchor; exact table coefficient"),
        ],
    )
)

# ------------------------------------------------------------
# 2. Maoz et al. 2018
# Keep log trade only.
# ------------------------------------------------------------
study_models.append(
    make_study_model(
        study_name="Maoz et al. 2018",
        hero_vars=["log trade"],
        node_map={
            canon("log trade"): "Trade",
            canon("relative capabilities"): "Capabilities",
            canon("joint democracy"): "Democracy",
            "Conflict": "Conflict",
        },
    )
)

# ------------------------------------------------------------
# 3. Miller & Gibler 2011
# Keep territorial dispute only.
# ------------------------------------------------------------
study_models.append(
    make_study_model(
        study_name="Miller & Gibler 2011",
        hero_vars=["territorial dispute"],
        node_map={
            canon("territorial dispute"): "TerritorialDispute",
            canon("capability ratio"): "Capabilities",
            canon("joint democracy"): "Democracy",
            canon("state a democracy"): "Democracy",
            canon("state b democracy"): "Democracy",
            "Conflict": "Conflict",
        },
    )
)

# ------------------------------------------------------------
# 4. Bell and Johnson 2015
# This is one of the few dense studies worth using for overlap.
# Keep hero = peace years.
# Add exact same-study anchors only.
# ------------------------------------------------------------
study_models.append(
    make_study_model(
        study_name="Bell and Johnson 2015",
        hero_vars=["peace years"],
        node_map={
            canon("peace years"): "LeaderHorizon",
            canon("joint democracy"): "Democracy",
            canon("expected power shift"): "CommitmentProblem",
            canon("foreign policy similarity"): "PreferenceSimilarity",
            "Conflict": "Conflict",
        },
        extra_moments=[
            ("expected shift in the distribution of power", "control", "same-study anchor"),
            ("similarity in foreign policy interests", "control", "same-study anchor; similarity mapped into PreferenceSimilarity node"),
        ],
    )
)

# ------------------------------------------------------------
# 5. Bas and Schub 2016
# Keep joint democracy as hero if you want a democracy anchor.
# Keep uncertainty terms separate if used at all.
# ------------------------------------------------------------
study_models.append(
    make_study_model(
        study_name="Bas and Schub 2016",
        hero_vars=["joint democracy"],
        node_map={
            canon("joint democracy"): "Democracy",
            canon("dyadic uncertainty"): "CommitmentProblem",
            canon("system uncertainty"): "CommitmentProblem",
            "Conflict": "Conflict",
        },
        extra_moments=[
            ("dyadic uncertainty", "control", "same-study anchor"),
            ("system uncertainty", "control", "same-study anchor"),
        ],
    )
)

# ------------------------------------------------------------
# 6. Mousseau, Hegre, and Oneal 2003
# Keep dependency as hero.
# Add only exact same-study overlap anchors.
# ------------------------------------------------------------
study_models.append(
    make_study_model(
        study_name="Mousseau, Hegre, and Oneal 2003",
        hero_vars=["dependencyl"],
        node_map={
            canon("dependencyl"): "Trade",
            canon("ln(capability ratio)"): "Capabilities",
            canon("democracyl"): "Democracy",
            canon("brevity of peace (5-year half-life)"): "LeaderHorizon",
            "Conflict": "Conflict",
        },
        extra_moments=[
            ("ln(capability ratio)", "control", "same-study anchor"),
            ("democracyl", "control", "same-study anchor"),
            ("brevity of peace (5-year half-life)", "control", "same-study anchor"),
        ],
    )
)

# ------------------------------------------------------------
# 7. Kim 2018
# Use ONE territorial-rivalry coefficient, not both by default.
# They are too close conceptually and can overweight one paper.
# ------------------------------------------------------------
study_models.append(
    make_study_model(
        study_name="Kim 2018",
        hero_vars=["territorial rivalry (strategic)"],
        node_map={
            canon("territorial rivalry (strategic)"): "TerritorialDispute",
            "Conflict": "Conflict",
        },
    )
)

# ------------------------------------------------------------
# 8. Owsiak and Rider 2013
# Keep power parity as hero.
# ------------------------------------------------------------
study_models.append(
    make_study_model(
        study_name="Owsiak and Rider 2013",
        hero_vars=["power parity"],
        node_map={
            canon("power parity"): "CommitmentProblem",
            canon("joint democracy"): "Democracy",
            "Conflict": "Conflict",
        },
    )
)

# ------------------------------------------------------------
# 9. Haynes 2017
# Keep s-score only.
# Note: s-score is similarity/alignment, not literal distance.
# ------------------------------------------------------------
study_models.append(
    make_study_model(
        study_name="Haynes 2017",
        hero_vars=["s score"],
        node_map={
            canon("s score"): "PreferenceSimilarity",
            canon("sender democracy"): "Democracy",
            canon("target democracy"): "Democracy",
            canon("trade"): "Trade",
            "Conflict": "Conflict",
        },
    )
)

# ------------------------------------------------------------
# 10. Fuhrmann and Kreps 2010
# Keep foreign policy similarity only.
# ------------------------------------------------------------
study_models.append(
    make_study_model(
        study_name="Fuhrmann and Kreps 2010",
        hero_vars=["foreign policy similarity"],
        node_map={
            canon("foreign policy similarity"): "PreferenceSimilarity",
            canon("power ratio"): "Capabilities",
            canon("target's polity"): "Democracy",
            "Conflict": "Conflict",
        },
    )
)









# ============================================================
# STEP 4: RUN THE ENGINE
# ============================================================
if not study_models:
    print("\n[INFO] No StudyModel objects have been added yet.")
    print("Fill in study_models with one or more StudyModel(...) entries,")
    print("then run this file again.\n")
else:
    engine = GodsDagEngine(
        dag=dag,
        studies=study_models,
        lambda_ridge=0.001,
        n_bootstrap=100,
    )

    results = engine.run()

    optimizer = results["optimizer"]
    diag = results["diagnostics"]
    counts = results["counts"]

    print("\n[ENGINE DIAGNOSTICS]")
    print("-" * 50)
    print(f"  Converged:        {optimizer.success}")
    print(f"  Final loss:       {optimizer.fun:.6f}")
    print(f"  Unknowns:         {diag['n_params']}")
    print(f"  Moments:          {diag['n_moments']}")
    print(f"  Study clusters:   {counts['study_clusters']}")
    print(f"  Jacobian rank:    {diag['jacobian_rank']}")
    print(f"  Condition number: {diag['condition_number']:.2f}")

    print("\n[PREDICTION] Example conflict probability")
    print("-" * 50)

    scenario = {
        "Capabilities": 0.0,
        "Trade": -1.0,
        "TerritorialDispute": +1.0,
        "CommitmentProblem": +1.0,
        "PreferenceSimilarity": +1.0,
        "LeaderHorizon": -1.0,
        "Democracy": 0.0,
        "GDP": 0.0,
    }

    pred = engine.predict_conflict_probability(scenario, base_rate=0.04)

    print(f"  Scenario: {scenario}")
    print(f"  Base rate:              {pred['base_rate']:.3f} ({pred['base_rate']*100:.1f}%)")
    print(f"  Log-odds shift:         {pred['log_odds_shift']:+.4f}")
    print(f"  Predicted probability:  {pred['predicted_probability']:.4f} ({pred['predicted_probability']*100:.1f}%)")


GOD'S DAG ENGINE — RAO CORP (v2)

[1] CLASSIFYING MOMENTS
----------------------------------------
  ✓ Gartzke Weisiger 2013 :: capability ratio: SAINT — All backdoor paths blocked
  ✓ Gartzke Weisiger 2013 :: dyadic difference: SAINT — All backdoor paths blocked
  ✓ Gartzke Weisiger 2013 :: systemic difference: SAINT — All backdoor paths blocked
  ✓ Maoz et al. 2018 :: log trade: SAINT — All backdoor paths blocked
  ✓ Miller & Gibler 2011 :: territorial dispute: SAINT — All backdoor paths blocked
  ✓ Bell and Johnson 2015 :: peace years: SAINT — All backdoor paths blocked
  ✓ Bell and Johnson 2015 :: expected power shift: SAINT — All backdoor paths blocked
  ✓ Bell and Johnson 2015 :: foreign policy similarity: SAINT — All backdoor paths blocked
  ~ Bas and Schub 2016 :: joint democracy: SINNER — 1 open backdoor path(s)
  ✓ Bas and Schub 2016 :: dyadic uncertainty: SAINT — All backdoor paths blocked
  ✓ Bas and Schub 2016 :: system uncertainty: SAINT — All backdoor paths blocked
  ✓ M